In [3]:
%pip install opencv-python numpy shapely matplotlib scikit-image
%pip install --upgrade tdmclient

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached tdmclient-0.1.21-py3-none-any.whl.metadata (78 kB)
  Using cached ifaddr-0.2.0-py3-none-any.whl.metadata (4.9 kB)
Using cached tdmclient-0.1.21-py3-none-any.whl (109 kB)
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 9.2 MB/s  0:00:00
Using cached ifaddr-0.2.0-py3-none-any.whl (12 kB)

   ---------- ----------------------------- 1/4 [zeroconf]
   ---------- ----------------------------- 1/4 [zeroconf]
   -------------------- ------------------- 2/4 [websockets]
   -------------------- ------------------- 2/4 [websockets]
   -------------------- ------------------- 2/4 [websockets]
   -------------------- ------------------- 2/4 [websockets]
   ------------------------------ --------- 3/4 [tdmclient]
   ------------------------------ --------- 3/4 [tdmclient]
   ---------------------------------------- 4/4 [tdmclient]

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import tdmclient.notebook
await tdmclient.notebook.start()

@tdmclient.notebook.sync_var
def get_sensor_data():
    global prox_horizontal
    return prox_horizontal.copy()

@tdmclient.notebook.sync_var
def set_speed(left,right):
    global motor_left_target, motor_right_target
    motor_left_target = left
    motor_right_target = right


In [1]:
import cv2
import numpy as np
from shapely.geometry import Polygon

from vision import (initialize_camera, initialize_vision, detect_thymio, draw_thymio,
                    GridNotDetectedError, camera_id, camera_width, fps,
                    warmup_frames, grid_side_mm)
from global_navigation import (build_free_space, compute_path,
                               draw_obstacles, draw_path, draw_legend)
from motion_control import motion

ARENA_W = ARENA_H = grid_side_mm   # 700 mm, from vision (origin top left corner)


def init_obstacles(frame):
    """Run the vision, inflate the obstacles and show them on the frame.

    Returns H (pixels -> mm), to_camera (mm -> pixels), obstacles, inflated obstacles, goal.
    """
    H, obstacles_mm, goal_mm = initialize_vision(frame)
    to_camera = np.linalg.inv(H)
    obstacles = [Polygon(v) for v in obstacles_mm]
    _, inflated_obstacles, _ = build_free_space(obstacles, ARENA_W, ARENA_H)
    cv2.imshow("Vision: inflated obstacles",
               draw_legend(draw_obstacles(frame.copy(), obstacles, inflated_obstacles, to_camera)))
    return H, to_camera, obstacles, inflated_obstacles, goal_mm


def main():
    cap = initialize_camera(camera_id, camera_width, fps, warmup_frames)
    dictionary = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
    detector = cv2.aruco.ArucoDetector(dictionary)
    frame = None

    try:
        # 1. Vision: grid, pieces, best move (once)
        ret, frame = cap.read()
        if not ret:
            raise RuntimeError("Cannot read the camera image")
        H, to_camera, obstacles, inflated_obstacles, goal_mm = init_obstacles(frame)

        waypoints = samples = None
        i = 0
        state = 0  # 0: goal tracking, 1: obstacle avoidance
        v_left = 0  # left wheel speed (mm/s)
        v_right = 0  # right wheel speed (mm/s)
        while True:
            ret, frame = cap.read()
            if not ret:
                raise RuntimeError("Cannot read the camera image")

            # 2. Thymio pose (every frame)
            thymio = detect_thymio(frame, H, detector)

            # 3. Global navigation: once, as soon as the Thymio is seen
            if waypoints is None and thymio is not None:
                waypoints, samples = compute_path(obstacles, thymio, goal_mm, ARENA_W, ARENA_H)

            # 4. Display: inflated obstacles, path, Thymio pose and orientation
            view = draw_obstacles(frame.copy(), obstacles, inflated_obstacles, to_camera)
            view = draw_path(view, waypoints, samples, to_camera)
            draw_thymio(view, thymio, to_camera)   # draws in place, label at the top left
            cv2.imshow("Global navigation", draw_legend(view))

            # Read proximity sensors
            capteurs = get_sensor_data()
            left_cmd, right_cmd, i = motion(samples, thymio, v_left, v_right, i, state, capteurs)
            # Send commands to motors
            set_speed(left_cmd, right_cmd)

            if i == len(samples):
                print("Goal reached! press 'r' to play next move, or 'q' to quit.")

            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):                 # quit
                break
            if key == ord("p") and thymio is not None:   # replan (e.g. after a kidnapping)
                waypoints, samples = compute_path(obstacles, thymio, goal_mm, ARENA_W, ARENA_H)
            if key == ord("r"):                 # rerun the vision (pieces moved)
                try:
                    H, to_camera, obstacles, inflated_obstacles, goal_mm = init_obstacles(frame)
                    waypoints = samples = None
                except GridNotDetectedError as error:
                    print(error)

    except GridNotDetectedError as error:
        print(error)
        cv2.imshow("Grid not detected (press a key to quit)", frame)
        cv2.waitKey(0)
    finally:
        cap.release()
        cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

Caméra initialisée
Résolution : 1920 x 1080
FPS configurés : 30.0
Path: 669 mm, 2 waypoints, 24 sampled points


NameError: name 'get_sensor_data' is not defined